cd Desktop/Coding\ project/StockAnalysis ; git add . ; git commit -m "In progress" ; git push origin main

In [1]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import get_current_date, get_df, get_infix, get_volume5m_data, generate_end_dates, merge_stocks, stock_market
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from scipy.stats import kendalltau, linregress, pearsonr, spearmanr, ttest_ind
from stock_screener import check_conds_tech, check_conds_fund, get_stock_info, stoploss_target
from technicals import *
from tqdm import tqdm

In [2]:
# Start of the program
start = dt.datetime.now()

# Variables
HKEX_all = False
NASDAQ_all = True
period_hk = 60 # Period for HK stocks
period_us = 252 # Period for US stocks
RS = 90
factors = [1, 1, 1]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix(index_name, index_dict, NASDAQ_all)

# Get the current date
current_date = get_current_date(start, index_name)

# Define the result folder
result_folder = "Result"

In [ ]:
# Choose the stocks
stocks = ["1810.HK", "3888.HK"]
for stock in stocks:
    df = get_df(stock, current_date)
    plot_close(stock, df)
    plot_volatility(stock, df)
    plot_ADX(stock, df)
    plot_MACD(stock, df, show=120)
    plot_MFI_RSI(stock, df)
    if stock.endswith(".HK"):
        plot_stocks(["^HSI", stock], current_date)
    else:
        plot_stocks(["^GSPC", stock], current_date)

In [ ]:
# Get the stop loss and target price of a stock
stock = "DOCU"
size = 15
industry = "Tech/Software"
entry_date = "2024-12-13"
entry_date_fmt = dt.datetime.strptime(entry_date, "%Y-%m-%d").strftime("%d-%m-%y")
df = get_df(stock, current_date)
current_close = df["Close"].iloc[-1]
for i in [5, 10, 20, 50]:
    df[f"SMA {i}"] = SMA(df, i)
entry = df["SMA 10"].iloc[-1]
entry = 95.7
stoploss, stoploss_pct, target, target_pct = stoploss_target(stock, entry, entry_date)
print(f"Plan for {stock}.")
print(f"Current close: {round(current_close, 2)}.")
print(f"{stock} {size}% ({industry}) Entry {entry} ({entry_date_fmt}) SL {stoploss} ({stoploss_pct}%) TP {target} ({target_pct}%)")

In [23]:
# Get the price data of QQQ, TQQQ, and VIX
qqq_df = get_df("^IXIC", current_date)
tqqq_df = get_df("TQQQ", current_date)
vix_df = get_df("^VIX", current_date)

# Filter qqq_df to match the starting date of TQQQ
qqq_df_tqqq = qqq_df[qqq_df.index >= tqqq_df.index[0]]

# Calculate the CAGR of TQQQ
CAGR_tqqq = (tqqq_df["Close"].iloc[-1] / tqqq_df["Close"].iloc[0])**(1 / (len(tqqq_df) / 252)) - 1

# Calculate the CAGR of QQQ
CAGR_qqq = (qqq_df_tqqq["Close"].iloc[-1] / qqq_df_tqqq["Close"].iloc[0])**(1 / (len(qqq_df_tqqq) / 252)) - 1

# Calculate the CAGR ratio of TQQQ to QQQ
tqqq_factor = CAGR_tqqq / CAGR_qqq

# Calculate the percentage change of QQQ
qqq_df["Percent Change"] = qqq_df["Close"].pct_change()

# Calculate the closing prices of TQQQ
qqq_df["TQQQ Percent Change"] = qqq_df["Percent Change"] * tqqq_factor
qqq_df["TQQQ Close"] = (1 + qqq_df["TQQQ Percent Change"]).cumprod()

# Create the mock TQQQ dataframe
tqqq_mock_df = qqq_df[["TQQQ Close", "TQQQ Percent Change"]].copy()

# Copy the index from qqq_df
tqqq_mock_df.index = qqq_df.index

# Scale the mock TQQQ closing prices to match the most recent TQQQ close
tqqq_mock_df["TQQQ Close"] = tqqq_mock_df["TQQQ Close"] * (tqqq_df["Close"].iloc[-1] / tqqq_mock_df["TQQQ Close"].iloc[-1])
tqqq_mock_df.rename(columns={"TQQQ Close": "Close", "TQQQ Percent Change": "Percent Change"}, inplace=True)

In [ ]:
show = 252 * 3
stocks = ["GC=F", "SI=F", "HG=F"]
metal_df = merge_stocks(stocks, current_date)
metal_df["Gold/Silver Ratio"] = metal_df["Close (GC=F)"] / metal_df["Close (SI=F)"]
metal_df["Gold/Copper Ratio"] = metal_df["Close (GC=F)"] / metal_df["Close (HG=F)"]
metal_df = calculate_zscore(metal_df, ["Gold/Silver Ratio", "Gold/Copper Ratio"], 252)

# Restrict the dataframe
metal_df = metal_df[- show:]

# Create a figure with three subplots, one for the metal prices, one for the ratios, one for the ratios z-score
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 8), gridspec_kw={"height_ratios": [3, 1, 1]}, sharex=True)

# Plot the metal prices on the first subplot
close_goldfirst = metal_df["Close (GC=F)"].iloc[0]
close_silverfirst = metal_df["Close (SI=F)"].iloc[0]
close_copperfirst = metal_df["Close (HG=F)"].iloc[0]
ax1.plot(100 / close_goldfirst * metal_df["Close (GC=F)"], label="Gold (scaled)", color="gold")
ax1.plot(100 / close_silverfirst * metal_df["Close (SI=F)"], label="Silver (scaled)", color="silver")
ax1.plot(100 / close_copperfirst * metal_df["Close (HG=F)"], label="Copper (scaled)", color="peru")

# Set the label of the first subplot
ax1.set_ylabel("Price")

# Set the x limit of the first subplot
ax1.set_xlim(metal_df.index[0], metal_df.index[-1])

# Plot the ratios on the second subplot
goldsilver_ratio_first = metal_df["Gold/Silver Ratio"].iloc[0]
goldcopper_ratio_first = metal_df["Gold/Copper Ratio"].iloc[0]
ax2.plot(100 / goldsilver_ratio_first * metal_df["Gold/Silver Ratio"], color="silver")
ax2.plot(100 / goldcopper_ratio_first * metal_df["Gold/Copper Ratio"], color="peru")

# Set the y label of the second subplot
ax2.set_ylabel("Ratio wrt Gold")

# Plot the ratios z-score on the third subplot
ax3.plot(metal_df["Gold/Silver Ratio Z-Score"], color="silver")
ax3.plot(metal_df["Gold/Copper Ratio Z-Score"], color="peru")
ax3.axhline(y=2, linestyle="dotted", label="Undervalued", color="green")
ax3.axhline(y=-2, linestyle="dotted", label="Overvalued", color="red")

# Set the y label of the third subplot
ax3.set_ylabel("Ratio z-score")

# Set the x label
plt.xlabel("Date")

# Set the title
plt.suptitle(f"Metal prices comparison")

# Combine the legends and place them at the top subplot
handles, labels = ax1.get_legend_handles_labels()
handles += ax3.get_legend_handles_labels()[0]
labels += ax3.get_legend_handles_labels()[1]
ax1.legend(handles, labels)

# Adjust the spacing between subplots
plt.tight_layout()

# Save the plot
plt.savefig("Result/Figure/metalcf.png", dpi=300)    

# Show the plot
plt.show()